# EMA 6938 - Data Science for Materials
## Week 10 Lab Notebook: Feature Engineering, Model Tuning, and Structure-Property Mapping

**Name:** *(your name here)*  
**Date:** *(date)*  
**Kernel:** Python (matds)

---

**Chapters:** Sandfeld Ch. 16  
**Format:** Parts A-F in-class - due **Sunday 11:59 PM**   
**Dataset:** `data/week10_mp_mapping.csv` (instructor-provided)

---

### How to use this notebook
- **Demo cells** (`# LECTURE DEMO`) reproduce examples from the lecture. Run them, understand them.
- **Task cells** (`# YOUR CODE HERE`) require you to write code.
- **Reflection cells** require written markdown answers. Replace the italic placeholder text.

This notebook has 7 parts:

| Part | Title | Connects to |
|------|-------|-------------|
| A | Load & Inspect | Lecture Segment 1 |
| B | Leakage-Aware Split | Lecture Segment 2 |
| C | Tune a Random Forest | Lecture Segment 4 |
| D | Feature Importance & Selection | Lecture Segment 1 |
| E | UMAP Projection | Lecture Segment 5 |
| F | White-Space Discovery | Lecture Segment 5 |
| G | Reflection & Midterm Bridge | All segments |

This notebook is also the template for the midterm reproduce-and-extend task.

---

**Submission:** Upload this `.ipynb` file to Canvas. Run `Kernel → Restart & Run All` before submitting to confirm all cells execute cleanly.

> **AI tool disclosure:** If you used any AI coding assistant (GitHub Copilot, ChatGPT, etc.) while completing this notebook, describe briefly which tool, for what purpose, and what you verified yourself. Delete this line if no AI tools were used.

In [ ]:
# Cell 0 — Environment check
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (GroupKFold, cross_val_score,
    RandomizedSearchCV, learning_curve, train_test_split)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from pymatgen.core import Composition
import warnings; warnings.filterwarnings('ignore')
from sklearn.exceptions import SkipTestWarning
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

try:
    import umap
    print("umap-learn OK")
except ImportError:
    print("WARNING: umap-learn not found - run: pip install umap-learn")

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
print("Core imports OK")

---
## Part A - Load & Inspect

### A1: Load the dataset

In [ ]:
# Cell A1
# LECTURE DEMO
df = pd.read_csv('data/week10_mp_mapping.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns[:8].tolist()} ...")
print(f"\nBand gap - mean: {df['band_gap'].mean():.2f} eV, std: {df['band_gap'].std():.2f}")
print(f"Formation energy - mean: {df['Ef_eV_atom'].mean():.3f} eV/atom")
print(f"\nCrystal systems:\n{df['crystal_system'].value_counts().head(6)}")

### A2: Set up the feature matrix X and target y

In [ ]:
# Cell A2
# LECTURE DEMO
non_feature_cols = ['mp_id','formula','band_gap','Ef_eV_atom','volume_A3',
                    'density_g_cm3','crystal_system','spacegroup','composition',
                    'is_metallic','is_insulating','is_theoretical']
feature_cols = [c for c in df.columns
                if c not in non_feature_cols and df[c].dtype in ['float64','float32']]
print(f"MAGPIE features: {len(feature_cols)}")

X_all = df[feature_cols].values
y_all = df['band_gap'].values

mask  = ~(np.isnan(X_all).any(axis=1) | np.isnan(y_all))
X_all = X_all[mask]; y_all = y_all[mask]
df_clean = df[mask].reset_index(drop=True)
print(f"Final shape: {X_all.shape}  ({(~mask).sum()} rows dropped)")

### A3: Compute reduced formula for GroupKFold groups

> GroupKFold needs a group label for each sample. We use the reduced formula so that all polymorphs (e.g. TiO2 rutile and anatase) always end up in the same fold.

In [ ]:
# Cell A3
# LECTURE DEMO
df_clean['reduced_formula'] = df_clean['formula'].apply(
    lambda f: Composition(f).reduced_formula)

n_unique = df_clean['reduced_formula'].nunique()
n_total  = len(df_clean)
print(f"Total entries:          {n_total:,}")
print(f"Unique reduced formulas:{n_unique:,}")
print(f"Avg entries per formula:{n_total/n_unique:.1f}")
print(f"\nMost frequent reduced formulas:")
print(df_clean['reduced_formula'].value_counts().head(8))

---
## Part B - Leakage-Aware Split

### B1: GroupKFold split

In [ ]:
# Cell B1
# LECTURE DEMO
groups = df_clean['reduced_formula'].values
gkf    = GroupKFold(n_splits=5)

# Extract fold 0 for Parts C–F
folds = list(gkf.split(X_all, y_all, groups))
train_idx, test_idx = folds[0]

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
groups_train    = groups[train_idx]

print(f"Fold 0 - Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Train mean band_gap: {y_train.mean():.3f} eV")
print(f"Test  mean band_gap: {y_test.mean():.3f} eV")

# Confirm no composition overlap between train and test
train_formulas = set(df_clean.iloc[train_idx]['reduced_formula'])
test_formulas  = set(df_clean.iloc[test_idx]['reduced_formula'])
overlap = train_formulas & test_formulas
print(f"\nComposition overlap between train and test: {len(overlap)} formulas")
print("(Should be 0 - that is the point of GroupKFold)")

### B2: Compare random split vs. GroupKFold - the leakage gap

In [ ]:
# Cell B2
# LECTURE DEMO
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("rf",     RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED))
])

# Random 5-fold CV
r2_random = cross_val_score(pipe, X_all, y_all, cv=5, scoring="r2")

# GroupKFold — no composition overlap
r2_group  = cross_val_score(pipe, X_all, y_all,
                             cv=GroupKFold(5), groups=groups, scoring="r2")

print("=" * 50)
print(f"Random 5-fold CV R2:  {r2_random.mean():.3f} ± {r2_random.std():.3f}")
print(f"GroupKFold R2:        {r2_group.mean():.3f}  ± {r2_group.std():.3f}")
print(f"Inflation from random split: {r2_random.mean() - r2_group.mean():.3f}")
print("=" * 50)
print("\nThe gap above is the composition leakage component.")
print("GroupKFold is the honest estimate of generalisation to new compositions.")

### B3: Scale within the training fold only

In [ ]:
# Cell B3
# LECTURE DEMO
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)          # transform only on test

print(f"X_train_scaled mean: {X_train_scaled.mean():.4f}  std: {X_train_scaled.std():.4f}")
print(f"X_test_scaled  mean: {X_test_scaled.mean():.4f}  (not guaranteed ~0)")

---
## Part C - Hyperparameter Tuning

### C1: Default RF baseline

In [ ]:
# Cell C1
# LECTURE DEMO
rf_default = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED)
rf_default.fit(X_train_scaled, y_train)
y_pred_default = rf_default.predict(X_test_scaled)

r2_def  = r2_score(y_test, y_pred_default)
mae_def = mean_absolute_error(y_test, y_pred_default)
rmse_def= np.sqrt(mean_squared_error(y_test, y_pred_default))

print("Default RF:")
print(f"  R2:   {r2_def:.3f}")
print(f"  MAE:  {mae_def:.3f} eV")
print(f"  RMSE: {rmse_def:.3f} eV")

### C2: RandomizedSearchCV with GroupKFold inner CV

In [ ]:
# Cell C2 — Hyperparameter tuning
# LECTURE DEMO
# Set TUNER to choose your method:
#   'random'  — RandomizedSearchCV (sklearn, always available)
#   'optuna'  — Bayesian optimisation via Optuna (pip install optuna)
#
# Both produce the same interface — C3 works unchanged either way.

TUNER = 'random'   # ← change to 'optuna' to use Bayesian optimisation

if TUNER == 'random':
    from sklearn.model_selection import RandomizedSearchCV

    param_dist = {
        "rf__n_estimators":     [100, 200, 300, 500],
        "rf__max_depth":        [None, 5, 10, 20],
        "rf__min_samples_leaf": [1, 2, 5],
        "rf__max_features":     ["sqrt", "log2", 0.3],
    }

    rscv = RandomizedSearchCV(
        pipe, param_dist, n_iter=30,
        cv=GroupKFold(n_splits=5),
        scoring="r2", n_jobs=-1, random_state=SEED, verbose=0
    )
    rscv.fit(X_train, y_train, groups=groups_train)

    best_model    = rscv
    best_params   = rscv.best_params_
    best_cv_score = rscv.best_score_
    print("Method: RandomizedSearchCV (30 random trials)")

elif TUNER == 'optuna':
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except ImportError:
        raise ImportError(
            "Optuna not installed. Run:  pip install optuna\n"
            "Or set TUNER = 'random' above to use RandomizedSearchCV instead."
        )
    from sklearn.model_selection import cross_val_score

    def objective(trial):
        params = {
            "rf__n_estimators":     trial.suggest_int("n_estimators", 100, 500, step=100),
            "rf__max_depth":        trial.suggest_int("max_depth", 3, 20),
            "rf__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "rf__max_features":     trial.suggest_float("max_features", 0.1, 0.9),
        }
        pipe.set_params(**params)
        scores = cross_val_score(
            pipe, X_train, y_train,
            cv=GroupKFold(n_splits=5),
            groups=groups_train,
            scoring="r2", n_jobs=-1
        )
        return scores.mean()

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=30, show_progress_bar=False)

    best_trial_params = {f"rf__{k}": v for k, v in study.best_params.items()}
    pipe.set_params(**best_trial_params)
    pipe.fit(X_train, y_train)

    class _OptunaTuned:
        def __init__(self, pipeline, params, score):
            self.best_params_ = params
            self.best_score_  = score
            self._pipe        = pipeline
        def predict(self, X): return self._pipe.predict(X)

    best_model    = _OptunaTuned(pipe, best_trial_params, study.best_value)
    best_params   = best_trial_params
    best_cv_score = study.best_value
    print("Method: Optuna Bayesian optimisation (30 trials, TPE sampler)")

else:
    raise ValueError(f"TUNER must be 'random' or 'optuna', got {TUNER!r}")

print(f"Best params:  {best_params}")
print(f"Best CV R\u00b2:   {best_cv_score:.3f}")

### C3: Evaluate tuned RF on test fold

In [ ]:
# Cell C3
# LECTURE DEMO
y_pred_tuned = best_model.predict(X_test)
r2_tuned   = r2_score(y_test, y_pred_tuned)
mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

comparison = pd.DataFrame({
    'Model':  ['Default RF', 'Tuned RF (RandomizedSearch)'],
    'R2':     [round(r2_def,3),   round(r2_tuned,3)],
    'MAE':    [round(mae_def,3),  round(mae_tuned,3)],
    'RMSE':   [round(rmse_def,3), round(rmse_tuned,3)],
})
print(comparison.to_string(index=False))

# Prediction scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, preds, title in zip(axes,
    [y_pred_default, y_pred_tuned],
    ['Default RF', 'Tuned RF']):
    ax.scatter(y_test, preds, s=8, alpha=0.5, color='#0D9488')
    lim = [min(y_test.min(), preds.min())-0.2, max(y_test.max(), preds.max())+0.2]
    ax.plot(lim, lim, 'k--', lw=0.8)
    ax.set_xlabel('Actual band gap (eV)'); ax.set_ylabel('Predicted (eV)')
    ax.set_title(f'{title}  $R^2$={r2_score(y_test,preds):.3f}')
plt.tight_layout(); plt.savefig('C3_prediction_scatter.png', dpi=150); plt.show()

### C4: Task - Learning curve for tuned RF

In [ ]:
# Cell C4
# TASK CELL
# YOUR CODE: use sklearn.model_selection.learning_curve
# Plot train R2 and CV R2 vs. training set size (10%, 30%, 50%, 70%, 100%)
# cv = GroupKFold(3); groups = groups_train; scoring = "r2"
# Save as C4_learning_curve.png
# What regime (high bias / high variance) does the curve show?

**Reflection C4 - fill in this cell:**

Looking at the learning curve: does the model appear to be in a high-bias or high-variance regime? What would you do to improve performance given what the curve shows?

*Your answer here*

---
## Part D - Feature Importance & Selection

### D1: Top 20 feature importances

In [ ]:
# Cell D1 — Extract the trained RF from the best estimator (inside the Pipeline)
# LECTURE DEMO
best_rf = rscv.best_estimator_.named_steps['rf']
importances = pd.Series(best_rf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(8, 6))
top20.sort_values().plot(kind='barh', color='#0D9488', ax=ax)
ax.set_xlabel('Mean impurity decrease')
ax.set_title('Top 20 MAGPIE features - Tuned RF (Week 10)')
plt.tight_layout()
plt.savefig('D1_feature_importance.png', dpi=150)
plt.show()

print("Top 10 features:")
print(top20.head(10).round(4))

### D2: Correlation filter - remove redundant features

In [ ]:
# Cell D2
# LECTURE DEMO
X_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
corr_matrix = X_df.corr().abs()

# Upper triangle
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Columns to drop: those with |r| > 0.95 with any other column
cols_to_drop = [c for c in upper.columns if any(upper[c] > 0.95)]
print(f"Features removed by correlation filter (|r|>0.95): {len(cols_to_drop)}")
print(f"Features remaining: {len(feature_cols) - len(cols_to_drop)}")

feature_cols_filtered = [c for c in feature_cols if c not in cols_to_drop]
X_train_filt = X_train_scaled[:, [feature_cols.index(c) for c in feature_cols_filtered]]
X_test_filt  = X_test_scaled[:,  [feature_cols.index(c) for c in feature_cols_filtered]]

### D3: Task - RF on filtered feature set

In [ ]:
# Cell D3
# TASK CELL
# YOUR CODE: train the tuned RF (best_params from C2) on X_train_filt
# Evaluate on X_test_filt
# Add a row to the comparison table from C3
# Is the performance change (+ or -) worth the feature reduction?

**Reflection D3 - fill in this cell:**

After removing correlated features, did R2 increase, decrease, or stay the same? What does this tell you about the information in the removed features? Were they truly redundant or were they carrying independent signal?

*Your answer here*

---
## Part E - UMAP Projection

### E1: Fit UMAP on training data (PCA-10 first for speed)

In [ ]:
# Cell E1
# LECTURE DEMO
from sklearn.decomposition import PCA

# PCA-10 first for speed and noise reduction
pca10 = PCA(n_components=10, random_state=SEED)
X_train_pca = pca10.fit_transform(X_train_scaled)

print(f"PCA-10 cumulative variance: {pca10.explained_variance_ratio_.sum():.1%}")

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
embedding_train = reducer.fit_transform(X_train_pca)

np.save('E1_umap_train.npy', embedding_train)
print(f"Training embedding shape: {embedding_train.shape}")

### E2: UMAP colored by actual band gap

In [ ]:
# Cell E2
# LECTURE DEMO
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(embedding_train[:,0], embedding_train[:,1],
                c=y_train, cmap='plasma', vmin=0, vmax=8, s=4, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Band gap (eV) - actual')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('Structure-Property Map - Actual Band Gap')
plt.tight_layout()
plt.savefig('E2_umap_actual.png', dpi=150)
plt.show()

### E3: UMAP colored by model predictions

In [ ]:
# Cell E3 — Predict band_gap for all training points with the tuned RF
# LECTURE DEMO
y_train_pred = rscv.best_estimator_.predict(X_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, vals, title in zip(axes,
    [y_train, y_train_pred],
    ['Actual band gap (eV)', 'Predicted band gap (eV)']):
    sc = ax.scatter(embedding_train[:,0], embedding_train[:,1],
                    c=vals, cmap='plasma', vmin=0, vmax=8, s=4, alpha=0.7)
    plt.colorbar(sc, ax=ax, label=title)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.set_title(title)

plt.suptitle('Structure-Property Map - Actual vs. Predicted', fontsize=11)
plt.tight_layout()
plt.savefig('E3_umap_actual_vs_predicted.png', dpi=150)
plt.show()

print("Regions where actual and predicted differ strongly = where the model struggles most.")

---
## Part F - White-Space Discovery

### F1: Generate candidate compositions

In [ ]:
# Cell F1 — Generate candidate binary oxide compositions not in the training set
# Elements: common oxide-forming metals + main group
candidate_elements = [
    'Li','Na','K','Mg','Ca','Sr','Ba',
    'Al','Ga','In','Tl',
    'Si','Ge','Sn','Pb',
    'Ti','Zr','Hf','V','Nb','Ta','Cr','Mo','W',
    'Mn','Fe','Co','Ni','Cu','Zn'
]

candidates = []
for el in candidate_elements:
    for ox in [2, 3, 4]:  # MO, M2O3, MO2 stoichiometries
        if ox == 2:
            formula = f"{el}O"
        elif ox == 3:
            formula = f"{el}2O3"
        else:
            formula = f"{el}O2"
        try:
            comp = Composition(formula)
            rf   = comp.reduced_formula
            if rf not in set(df_clean['reduced_formula']):  # not in training set
                candidates.append({'formula': formula, 'reduced_formula': rf})
        except:
            pass

df_cand = pd.DataFrame(candidates).drop_duplicates('reduced_formula')
print(f"Generated {len(df_cand)} candidate compositions not in training data")
print(df_cand.head(10))

### F2: Featurise candidates and project onto UMAP

In [ ]:
# Cell F2
# LECTURE DEMO
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty

stc = StrToComposition(target_col_id='composition')
df_cand_feat = stc.featurize_dataframe(df_cand.copy(), 'formula', ignore_errors=True)
ep = ElementProperty.from_preset('magpie')
df_cand_feat = ep.featurize_dataframe(df_cand_feat, 'composition', ignore_errors=True)
df_cand_feat = df_cand_feat.dropna(subset=feature_cols)

print(f"Successfully featurised: {len(df_cand_feat)} candidates")

X_cand = df_cand_feat[feature_cols].values
X_cand_scaled = scaler.transform(X_cand)       # use training scaler
X_cand_pca    = pca10.transform(X_cand_scaled) # use training PCA
embedding_cand = reducer.transform(X_cand_pca) # project onto training UMAP
np.save('F2_umap_candidates.npy', embedding_cand)

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(embedding_train[:,0], embedding_train[:,1],
           c=y_train, cmap='plasma', vmin=0, vmax=8, s=4, alpha=0.5, label='Training data')
ax.scatter(embedding_cand[:,0], embedding_cand[:,1],
           c='#CBD5E1', s=40, marker='*', alpha=0.9, label='Candidates (new compositions)')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('Training Data + Candidate Compositions in Composition Space')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('F2_umap_candidates.png', dpi=150)
plt.show()

### F3: Flag isolated candidates - white-space identification

In [ ]:
# Cell F3
# LECTURE DEMO
from sklearn.neighbors import NearestNeighbors

# Distance from each candidate to its nearest training point (in PCA-10 space)
nn = NearestNeighbors(n_neighbors=1)
nn.fit(X_train_pca)
distances, _ = nn.kneighbors(X_cand_pca)
distances = distances.ravel()

# Flag candidates in sparse regions (distance > 95th percentile of training pairwise distances)
threshold = np.percentile(distances, 95)
isolated  = distances > threshold

print(f"Distance threshold (95th percentile): {threshold:.3f}")
print(f"Isolated candidates (in white space): {isolated.sum()} / {len(df_cand_feat)}")

df_cand_feat['nn_distance'] = distances
df_cand_feat['in_whitespace'] = isolated
print("\nTop 5 most isolated candidates:")
print(df_cand_feat.nlargest(5, 'nn_distance')[['formula','reduced_formula','nn_distance']])

### F4: Task - predict band gap for isolated candidates

In [ ]:
# Cell F4 — Predict band_gap for all candidates with the tuned RF
# TASk CELL
y_cand_pred = rscv.best_estimator_.predict(df_cand_feat[feature_cols].values)
df_cand_feat['predicted_band_gap'] = y_cand_pred

# Focus on the 3 most isolated candidates
top3_isolated = df_cand_feat.nlargest(3, 'nn_distance')[
    ['formula','reduced_formula','nn_distance','predicted_band_gap']]
print("Top 3 most isolated candidates with predicted band gap:")
print(top3_isolated.to_string(index=False))

# YOUR TASK: Answer in a markdown cell below —
# Are these predictions trustworthy?
# A candidate far from all training points is an extrapolation.
# What additional evidence would you need before taking this prediction seriously?

**Reflection F4 - fill in this cell:**

For the top 3 isolated candidates: are their predicted band gaps trustworthy? Explain the concept of interpolation vs. extrapolation in the context of the UMAP map, and describe what evidence (experimental or computational) would be needed to validate the prediction.

*Your answer here*

---
## Part G - Reflection & Midterm Bridge

### G1: R2 gap — composition leakage

In 3–4 sentences: What was the R2 gap between random 5-fold CV and GroupKFold in Part B? What does this gap mean physically in terms of what the model learned vs. what we want it to generalise to? Is a model that performs well on random CV but poorly on GroupKFold useful for materials discovery?

*Your answer here*

### G2: White-space region - scientific hypothesis

In 3–4 sentences: Identify the white-space region you find most scientifically interesting in your UMAP map. Name the candidate composition family it corresponds to. State one physical reason why new materials in that region might have useful band gap properties and one reason why the model's prediction there might be unreliable.

*Your answer here*

### G3: Midterm bridge

In 2–3 sentences: The pipeline you built today (fetch → featurise → GroupKFold tune → UMAP → overlay → white-space) is the template for the midterm. Which step will you modify for your "extend" component — and why did you choose that step?

*Your answer here*

---
## Day 2

> **This section is covered on Day 2**

### Demo 1 - GroupKFold vs. random split: quantifying the leakage gap

In [ ]:
# Cell DEMO 1 - Compare CV strategies across multiple folds
# LECTURE DEMO

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

rf_base = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)

# Random 5-fold CV
r2_random = cross_val_score(rf_base, X_all, y_all, cv=5, scoring='r2')

# GroupKFold by reduced formula
r2_group  = cross_val_score(rf_base, X_all, y_all,
                             cv=GroupKFold(5), groups=groups,
                             scoring='r2')

print(f"Random 5-fold CV:      mean $R^2$={r2_random.mean():.3f}  std={r2_random.std():.3f}")
print(f"GroupKFold CV:         mean $R^2$={r2_group.mean():.3f}   std={r2_group.std():.3f}")
print(f"Leakage gap:           {r2_random.mean() - r2_group.mean():.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([r2_random, r2_group], tick_labels=['Random 5-fold', 'GroupKFold (reduced formula)'])
ax.set_ylabel('$R^2$ per fold'); ax.set_title('CV strategy comparison - leakage gap')
ax.axhline(r2_random.mean(), color='#EF4444', ls='--', lw=1, alpha=0.6)
ax.axhline(r2_group.mean(),  color='#0D9488', ls='--', lw=1, alpha=0.6)
plt.tight_layout()
plt.savefig('Day2_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Demo 2 - White-space trustworthiness: extrapolation vs. interpolation

In [ ]:
# Cell DEMO 2 — Annotated white-space UMAP: trustworthy vs. extrapolation regions
# LECTURE DEMO
# Load UMAP embeddings from Part E (must have run Part E first)
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(embedding_train[:,0], embedding_train[:,1],
                    c='#CBD5E1', s=5, alpha=0.4, label='Training data')
cd = ax.scatter(embedding_cand[:,0], embedding_cand[:,1],
                   c='#EF4444', s=20, alpha=0.7,
                   marker='*', label='White-space candidates')

# Annotate trust regions manually
ax.text(embedding_train[:,0].mean(), embedding_train[:,1].mean(),
            'Interpolation (trustworthy)', fontsize=9,
            ha='center', color='#1C2B4A',
            bbox=dict(boxstyle='round', fc='#E3F2FD', alpha=0.8))

ax.set_title('UMAP - training data + white-space candidates', fontsize=11)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('Day2_whitespace_trust.png', dpi=150, bbox_inches='tight')
plt.show()

### Demo 3 - Midterm bridge: what a good reproduce-and-extend looks like
**Instructor-led discussion - no code cell**

The midterm assigns you to reproduce the key quantitative result of an ML-for-materials paper,
then extend it with one methodological improvement.

**The reproduce step:** Load or reconstruct the dataset. Implement the same feature set and model.
Apply the same CV strategy. Report the same metrics. Discuss any discrepancy between your result
and the paper's. Discrepancies are interesting, not failures.

**The extend step (choose one):**
- Change random CV to GroupKFold - quantify the leakage gap
- Add one structure-based feature the paper did not use
- Apply Lasso feature selection and report which features survive
- Produce a learning curve the paper did not show

**The metric that matters:** Your extension must change at least one reported number
(R2, MAE, n_features) and you must explain physically why it changed.

**Day 2 Discussion questions:**

1. From Demo 1: how large is the leakage gap between random CV and GroupKFold for your dataset? If a paper reports $R^2$=0.92 with random CV, what would you expect with GroupKFold?

2. From Demo 2: which white-space candidates are most isolated from the training data? Are model predictions in those regions trustworthy? What would you need to trust them?

3. Midterm planning: which of the four extension options (CV strategy / structure feature / Lasso / learning curve) is most relevant to the paper you have been assigned? Which one will give you the most interesting finding?

---
## Submission Checklist

Before submitting, confirm all cells have been executed:

- [ ] B2: Random vs. GroupKFold comparison printed (with inflation gap)
- [ ] C3: Comparison table (Default RF vs. Tuned RF) + prediction scatter saved
- [ ] C4: Learning curve task completed and reflection filled in
- [ ] D1: Feature importance plot saved
- [ ] D3: Filtered RF task completed and reflection filled in
- [ ] E2–E3: Both UMAP scatters saved (actual + predicted)
- [ ] F2: Candidate UMAP overlay saved
- [ ] F3: Isolated candidates identified and listed
- [ ] F4: Prediction trustworthiness reflection filled in
- [ ] G1–G3: All reflection cells answered
- [ ] All reflection cells answered (no placeholder text)
- [ ] AI disclosure note updated or deleted at the top of the notebook
- [ ] File renamed: `[LastName]_week10.ipynb`
**Final check:** Run `Kernel → Restart & Run All`. All cells must execute without errors before submitting.
**Submit via Canvas by Sunday 11:59 PM.**
**Midterm due: Sunday 11:59 PM - see the midterm notebook on GitHub.**